In [1]:
# import random
# from itertools import combinations
# from collections import Counter

# # ---- Card representation ----
# RANKS = list(range(2, 15))  # 2-14, 14 = Ace
# SUITS = ['s', 'h', 'd', 'c']
# DECK = [(r, s) for r in RANKS for s in SUITS]

# # ---- Hand evaluation (identical to main file) ----

# def best_hand_rank(cards):
#     best = None
#     for combo in combinations(cards, min(5, len(cards))):
#         rank = hand_rank(combo)
#         if best is None or rank > best:
#             best = rank
#     return best

# def hand_rank(cards):
#     ranks = sorted([r for r, s in cards], reverse=True)
#     suits = [s for r, s in cards]
#     counts = Counter(ranks)
#     rank_counts = sorted(counts.values(), reverse=True)

#     is_flush = len(set(suits)) == 1
#     is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

#     if set(ranks) == {14, 2, 3, 4, 5}:
#         is_straight = True
#         ranks = [5, 4, 3, 2, 1]
#         counts = Counter(ranks)

#     tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

#     if is_straight and is_flush:
#         return (8, tiebreaker)
#     if rank_counts[0] == 4:
#         return (7, tiebreaker)
#     if rank_counts[:2] == [3, 2]:
#         return (6, tiebreaker)
#     if is_flush:
#         return (5, tiebreaker)
#     if is_straight:
#         return (4, tiebreaker)
#     if rank_counts[0] == 3:
#         return (3, tiebreaker)
#     if rank_counts[:2] == [2, 2]:
#         return (2, tiebreaker)
#     if rank_counts[0] == 2:
#         return (1, tiebreaker)
#     return (0, tiebreaker)

# # ---- Candidate rules ----
# # rank_gte_7 through rank_gte_11 (7, 8, 9, 10, J)
# # plus suit_s_rank_gte_X for the same range (spades as representative suit)

# CANDIDATE_RULES = []

# for r in range(7, 12):
#     CANDIDATE_RULES.append((f'rank_gte_{r}', lambda c, r=r: c[0] >= r))

# SUIT = 's'
# for r in range(7, 12):
#     CANDIDATE_RULES.append((f'suit_{SUIT}_rank_gte_{r}',
#                              lambda c, r=r, s=SUIT: c[1] == s and c[0] >= r))

# # ---- Single game with fixed rule ----

# def play_game_fixed(rule_fn, n_picks=5, dealer_fill=8):
#     """Play one full game always using the same fixed rule."""
#     my_hand = []
#     dealer_hand = []
#     remaining_deck = list(DECK)

#     for _ in range(n_picks):
#         if not remaining_deck:
#             break

#         matching = [c for c in remaining_deck if rule_fn(c)]
#         if not matching:
#             break  # rule has no valid cards left, can't pick

#         random.shuffle(remaining_deck)
#         target_idx = next(i for i, c in enumerate(remaining_deck) if rule_fn(c))

#         my_hand.append(remaining_deck[target_idx])
#         dealer_hand += remaining_deck[:target_idx]
#         remaining_deck = remaining_deck[target_idx + 1:]

#     if len(my_hand) < 5:
#         return 0  # incomplete hand loses

#     random.shuffle(remaining_deck)
#     needed = max(0, dealer_fill - len(dealer_hand))
#     final_dealer = dealer_hand + remaining_deck[:needed]

#     return 1 if best_hand_rank(my_hand) > best_hand_rank(final_dealer) else 0

# # ---- Run all candidates ----

# def simulate_fixed(rule_fn, n_games=2000, n_picks=5, dealer_fill=8):
#     wins = sum(play_game_fixed(rule_fn, n_picks, dealer_fill) for _ in range(n_games))
#     return wins / n_games

# if __name__ == '__main__':
#     n_games = 2000
#     print(f"Simulating fixed rules over {n_games} games each...\n")
#     print(f"{'Rule':<25} {'Cards matched':<15} {'Win rate'}")
#     print("-" * 50)

#     results = []
#     for rule_name, rule_fn in CANDIDATE_RULES:
#         cards_matched = sum(1 for c in DECK if rule_fn(c))
#         win_rate = simulate_fixed(rule_fn, n_games=n_games)
#         results.append((rule_name, cards_matched, win_rate))
#         print(f"{rule_name:<25} {cards_matched:<15} {win_rate:.3f}")

#     print("-" * 50)
#     best = max(results, key=lambda x: x[2])
#     print(f"\nBest fixed rule: {best[0]} (win rate: {best[2]:.3f})")


flush forcing strat

In [2]:
# import random
# from itertools import combinations
# from collections import Counter

# # ---- Card representation ----
# RANKS = list(range(2, 15))  # 2-14, 14 = Ace
# SUITS = ['s', 'h', 'd', 'c']
# DECK = [(r, s) for r in RANKS for s in SUITS]

# # ---- Hand evaluation ----

# def best_hand_rank(cards):
#     best = None
#     for combo in combinations(cards, min(5, len(cards))):
#         rank = hand_rank(combo)
#         if best is None or rank > best:
#             best = rank
#     return best

# def hand_rank(cards):
#     ranks = sorted([r for r, s in cards], reverse=True)
#     suits = [s for r, s in cards]
#     counts = Counter(ranks)
#     rank_counts = sorted(counts.values(), reverse=True)

#     is_flush = len(set(suits)) == 1
#     is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

#     if set(ranks) == {14, 2, 3, 4, 5}:
#         is_straight = True
#         ranks = [5, 4, 3, 2, 1]
#         counts = Counter(ranks)

#     tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

#     if is_straight and is_flush:
#         return (8, tiebreaker)
#     if rank_counts[0] == 4:
#         return (7, tiebreaker)
#     if rank_counts[:2] == [3, 2]:
#         return (6, tiebreaker)
#     if is_flush:
#         return (5, tiebreaker)
#     if is_straight:
#         return (4, tiebreaker)
#     if rank_counts[0] == 3:
#         return (3, tiebreaker)
#     if rank_counts[:2] == [2, 2]:
#         return (2, tiebreaker)
#     if rank_counts[0] == 2:
#         return (1, tiebreaker)
#     return (0, tiebreaker)

# # ---- Candidate rules: suit_s_rank_gte_r for r in 2..7 ----

# SUIT = 's'
# CANDIDATE_RULES = []
# for r in range(2, 8):
#     CANDIDATE_RULES.append((
#         f'suit_{SUIT}_rank_gte_{r}',
#         lambda c, r=r, s=SUIT: c[1] == s and c[0] >= r
#     ))

# # ---- Single game with fixed rule ----

# def play_game_fixed(rule_fn, n_picks=5, dealer_fill=8):
#     """Play one full game always using the same fixed rule."""
#     my_hand = []
#     dealer_hand = []
#     remaining_deck = list(DECK)

#     for _ in range(n_picks):
#         if not remaining_deck:
#             break

#         matching = [c for c in remaining_deck if rule_fn(c)]
#         if not matching:
#             break  # no eligible cards left, forfeit remaining picks

#         random.shuffle(remaining_deck)
#         target_idx = next(i for i, c in enumerate(remaining_deck) if rule_fn(c))

#         my_hand.append(remaining_deck[target_idx])
#         dealer_hand += remaining_deck[:target_idx]
#         remaining_deck = remaining_deck[target_idx + 1:]

#     if len(my_hand) < 5:
#         return 0  # incomplete hand loses

#     random.shuffle(remaining_deck)
#     needed = max(0, dealer_fill - len(dealer_hand))
#     final_dealer = dealer_hand + remaining_deck[:needed]

#     return 1 if best_hand_rank(my_hand) > best_hand_rank(final_dealer) else 0

# # ---- Simulate all candidates ----

# def simulate_fixed(rule_fn, n_games=2000, n_picks=5, dealer_fill=8):
#     wins = sum(play_game_fixed(rule_fn, n_picks, dealer_fill) for _ in range(n_games))
#     return wins / n_games

# if __name__ == '__main__':
#     n_games = 2000
#     print(f"Simulating suit+rank fixed rules over {n_games} games each...\n")
#     print(f"{'Rule':<30} {'Cards in deck':<15} {'Win rate'}")
#     print("-" * 55)

#     results = []
#     for rule_name, rule_fn in CANDIDATE_RULES:
#         cards_in_deck = sum(1 for c in DECK if rule_fn(c))
#         win_rate = simulate_fixed(rule_fn, n_games=n_games)
#         results.append((rule_name, cards_in_deck, win_rate))
#         print(f"{rule_name:<30} {cards_in_deck:<15} {win_rate:.3f}")

#     print("-" * 55)
#     best = max(results, key=lambda x: x[2])
#     print(f"\nBest fixed rule: {best[0]} (win rate: {best[2]:.3f})")

In [3]:
import random
from itertools import combinations
from collections import Counter

# ---- Card representation ----
RANKS = list(range(2, 15))  # 2-14, 14 = Ace
SUITS = ['s', 'h', 'd', 'c']
DECK = [(r, s) for r in RANKS for s in SUITS]

# ---- Hand evaluation ----

def best_hand_rank(cards):
    best = None
    for combo in combinations(cards, min(5, len(cards))):
        rank = hand_rank(combo)
        if best is None or rank > best:
            best = rank
    return best

def hand_rank(cards):
    ranks = sorted([r for r, s in cards], reverse=True)
    suits = [s for r, s in cards]
    counts = Counter(ranks)
    rank_counts = sorted(counts.values(), reverse=True)

    is_flush = len(set(suits)) == 1
    is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

    if set(ranks) == {14, 2, 3, 4, 5}:
        is_straight = True
        ranks = [5, 4, 3, 2, 1]
        counts = Counter(ranks)

    tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

    if is_straight and is_flush:
        return (8, tiebreaker)
    if rank_counts[0] == 4:
        return (7, tiebreaker)
    if rank_counts[:2] == [3, 2]:
        return (6, tiebreaker)
    if is_flush:
        return (5, tiebreaker)
    if is_straight:
        return (4, tiebreaker)
    if rank_counts[0] == 3:
        return (3, tiebreaker)
    if rank_counts[:2] == [2, 2]:
        return (2, tiebreaker)
    if rank_counts[0] == 2:
        return (1, tiebreaker)
    return (0, tiebreaker)

# ---- Candidate rules ----

CANDIDATE_RULES = []

for r in range(3, 12):
    CANDIDATE_RULES.append((f'rank_gte_{r}', lambda c, r=r: c[0] >= r))

SUIT = 's'
for r in range(2, 11):
    CANDIDATE_RULES.append((f'suit_{SUIT}_rank_gte_{r}',
                             lambda c, r=r, s=SUIT: c[1] == s and c[0] >= r))

# ---- Single game with fixed rule ----

def play_game_fixed(rule_fn, n_picks=5, dealer_fill=8):
    my_hand = []
    dealer_hand = []
    remaining_deck = list(DECK)

    for _ in range(n_picks):
        if not remaining_deck:
            break
        matching = [c for c in remaining_deck if rule_fn(c)]
        if not matching:
            break
        random.shuffle(remaining_deck)
        target_idx = next(i for i, c in enumerate(remaining_deck) if rule_fn(c))
        my_hand.append(remaining_deck[target_idx])
        dealer_hand += remaining_deck[:target_idx]
        remaining_deck = remaining_deck[target_idx + 1:]

    if len(my_hand) < 5:
        return 0, 0, 0

    dealer_before = len(dealer_hand)
    random.shuffle(remaining_deck)
    needed = max(0, dealer_fill - len(dealer_hand))
    final_dealer = dealer_hand + remaining_deck[:needed]
    dealer_after = len(final_dealer)

    win = 1 if best_hand_rank(my_hand) > best_hand_rank(final_dealer) else 0
    return win, dealer_before, dealer_after

# ---- Simulate ----

def simulate_fixed(rule_fn, n_games=2000, n_picks=5, dealer_fill=8):
    wins = 0
    total_before = 0
    total_after = 0
    for _ in range(n_games):
        win, before, after = play_game_fixed(rule_fn, n_picks, dealer_fill)
        wins += win
        total_before += before
        total_after += after
    return wins / n_games, total_before / n_games, total_after / n_games

if __name__ == '__main__':
    n_games = 2000
    print(f"Simulating fixed rules over {n_games} games each...\n")
    print(f"{'Rule':<28} {'Matched':<9} {'Win rate':<10} {'Dealer (passing)':<18} {'Dealer (final)'}")
    print("-" * 78)

    results = []
    for rule_name, rule_fn in CANDIDATE_RULES:
        cards_matched = sum(1 for c in DECK if rule_fn(c))
        win_rate, avg_before, avg_after = simulate_fixed(rule_fn, n_games=n_games)
        results.append((rule_name, cards_matched, win_rate, avg_before, avg_after))
        print(f"{rule_name:<28} {cards_matched:<9} {win_rate:<10.3f} {avg_before:<18.2f} {avg_after:.2f}")

    print("-" * 78)
    best = max(results, key=lambda x: x[2])
    print(f"\nBest fixed rule: {best[0]} (win rate: {best[2]:.3f}, "
          f"avg dealer from passing: {best[3]:.2f}, avg dealer final: {best[4]:.2f})")

Simulating fixed rules over 2000 games each...

Rule                         Matched   Win rate   Dealer (passing)   Dealer (final)
------------------------------------------------------------------------------
rank_gte_3                   48        0.184      0.40               8.00
rank_gte_4                   44        0.188      0.89               8.00
rank_gte_5                   40        0.215      1.42               8.00
rank_gte_6                   36        0.239      2.15               8.00
rank_gte_7                   32        0.233      3.04               8.02
rank_gte_8                   28        0.268      4.09               8.10
rank_gte_9                   24        0.258      5.71               8.46
rank_gte_10                  20        0.289      7.60               9.30
rank_gte_11                  16        0.236      10.74              11.50
suit_s_rank_gte_2            13        0.341      13.97              14.30
suit_s_rank_gte_3            12        0.273   